# B05B — INbreast: ablation tanpa early stopping
Enam run: dua kontrol supervised dan empat pasangan Hinton KD. Setiap run dilatih ulang dari ImageNet dengan seed 42, 5 epoch head dan **25 epoch fine-tuning penuh**. Checkpoint terbaik tetap dipilih berdasarkan Macro-F1 validation, lalu Recall; checkpoint head tetap boleh terpilih.

Hasil B05 awal tetap terpisah. Teacher B04B hanya dimuat untuk inferensi, tidak dilatih ulang. Cache Teacher baru dibuat di folder ablation agar cache lama tidak ditulis ulang. Jalankan Cell 1–10 di Colab GPU. Evaluasi test tidak tersedia pada notebook diagnosis ini.


## Metode dan batas interpretasi
Untuk logit tunggal $z$, distribusi biner pada temperatur $T$ adalah
$q_T=[1-\sigma(z/T),\sigma(z/T)]$.

$$
L=(1-\alpha)\,\operatorname{mean}(w_y\,\mathrm{BCEWithLogits}(y,z_s))
+\alpha T^2\,\operatorname{mean}\!\left[D_{KL}(q_T^{teacher}\Vert q_T^{student})\right].
$$

- Bobot kelas $w_y=N/(2N_y)$ berasal hanya dari **train** dan diterapkan hanya pada loss label asli.
- Teacher memakai inference mode, bobot beku, dan tanpa gradient. Student belajar label asli + distribusi Teacher.
- `alpha=0.5`, `T=2.0` adalah **konfigurasi awal eksperimen**, bukan nilai optimal atau kewajiban dari paper. Pada ablation ini keduanya dipertahankan; jangan mengubahnya bersamaan dengan durasi training.
- Kontrol memakai `alpha=0`: loss supervised berbobot saja. Tidak ada feature KD, MoE, atau SSID-KD.
- Logit dihitung langsung dari kernel/bias classifier, bukan inversi probabilitas yang sudah dibulatkan. Single-output softmax **tidak digunakan**.
- Teacher melihat view citra yang sama dengan Student. Karena augmentasi hanya horizontal flip, cache dua view ekuivalen dengan inference Teacher online untuk augmentasi tersebut, dalam batas floating-point.
- Head 5 epoch → fine-tune 25 epoch penuh tanpa early stopping, Adam 1e-3 → 1e-5, batch 8, dropout 0.2, BN frozen/non-fused. Pemilihan: validation macro-F1, tie recall positif. Fine-tune dimulai dari best head; hasil head tetap dipilih bila fine-tune tidak membaik. Threshold utama **0.5**.
- Precision/Recall/F1 positif tetap metrik utama; macro-F1 dan specificity penting untuk CMMD yang tidak seimbang. Hasil seed 42 bersifat screening; tidak membuktikan signifikansi atau stabilitas antarseed.
- INbreast memakai target BI-RADS 1/2 vs 5/6, bukan label patologi yang identik dengan dataset lain. Pengelompokan pasien didukung audit identitas INbreast terpisah berdasarkan README sumber. MIAS hanya citra abnormal benign/malignant dan ukuran validation kecil.
- Split test B03 pernah dilihat. B05 menahan test selama tuning, tetapi **tidak menyebutnya holdout yang belum pernah dilihat**.

Acuan: [Hinton, Vinyals, & Dean (2015)](https://arxiv.org/abs/1503.02531), [TensorFlow Checkpoint](https://www.tensorflow.org/api_docs/python/tf/train/Checkpoint).

## Cell 1 — GPU Runtime and Google Drive
Run on a Colab GPU with the new runtime. The `tf-keras` package has been updated to match the TensorFlow version to ensure that the B04B checkpoint remains compatible. If the package has previously been imported with a different version, restart the runtime first.

In [ ]:
import os,sys,subprocess,importlib.metadata as metadata
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'

import tensorflow as tf
tf_minor='.'.join(tf.__version__.split('.')[:2])
try: installed=metadata.version('tf-keras')
except metadata.PackageNotFoundError: installed=''
if '.'.join(installed.split('.')[:2])!=tf_minor:
    if 'tf_keras' in sys.modules:raise RuntimeError('Restart runtime before changing tf-keras')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps',f'tf-keras~={tf_minor}.0'])

import tf_keras as keras
from pathlib import Path
from google.colab import drive
from IPython.display import display
gpus=tf.config.list_physical_devices('GPU')
if not gpus:raise RuntimeError('Select GPU runtime in Colab')
for gpu in gpus:tf.config.experimental.set_memory_growth(gpu,True)
keras.mixed_precision.set_global_policy('float32')
tf.config.experimental.enable_op_determinism()

print('TensorFlow:',tf.__version__,'tf-keras:',keras.__version__,'GPU:',gpus)
drive.mount('/content/drive')

## Cell 2 — Konfigurasi ablation
Default INbreast, seed 42, enam run, alpha 0,5 dan T=2. Early stopping dinonaktifkan. Folder output ablation baru; jangan arahkan ke folder B05 awal.


In [ ]:
RESEARCH=Path('/content/drive/MyDrive/Research')
DATASET='inbreast'  # cbis_ddsm | cmmd | inbreast | mias
TRAINING_SEED=42
TEACHER_SEED=42
STUDENTS_TO_RUN=['mobilenet_v2','nasnet_mobile']
TEACHERS_TO_RUN=['vgg16','densenet121']
BATCH_SIZE=8
TEACHER_BATCH_SIZE=8  # Same as batch inference B04B; audited separately from the Student batch.
HEAD_EPOCHS=5
FINETUNE_EPOCHS=25
EARLY_STOPPING=False
PATIENCE=5  # Tidak digunakan ketika EARLY_STOPPING=False.
HEAD_LEARNING_RATE=1e-3
FINETUNE_LEARNING_RATE=1e-5
TEMPERATURE=2.0
ALPHA=0.5
EVALUATE_TEST=False
DOWNLOAD_REPORT=True

B04A_REGISTRY=RESEARCH/'outputs/benchmark_4datasets/b04_preprocessing_v1/1dcb7ddadfe6/registry.json'
B04_RUN_ROOT=RESEARCH/'outputs/benchmark_4datasets/b04_ablation_runs_v1'
# Protocol ID ‘full_512’ from report B04B. Enter ‘None’ for automatic searches, which must be unique.
B04_PROTOCOL_FOLDERS={
    'cbis_ddsm':'ab66d3a7f2bd', 'cmmd':'87b9956404a1',
    'inbreast':'fb7385e67577', 'mias':'a3c269596e8c',
}
ORIGINAL_B05_ROOT=RESEARCH/'outputs/benchmark_4datasets/b05_hinton_kd_v1'
B05_ROOT=RESEARCH/'outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1'
if B05_ROOT.resolve()==ORIGINAL_B05_ROOT.resolve() or ORIGINAL_B05_ROOT.resolve() in B05_ROOT.resolve().parents:
    raise RuntimeError('Output ablation harus terpisah dari B05 awal')
CACHE_ROOT=B05_ROOT/'teacher_cache'
LOCAL_IMAGES=Path('/content/b05_images')

## Cell 3 — Trainer function and loss check
This long cell contains a standalone implementation. It does not need to be edited. Only the B04B compatibility helper is used, without calling the B04B training function.

In [ ]:
B04_COMPAT_SOURCE = "import os, io, gc, json, time, hashlib, zipfile, shutil, platform\n\nfrom pathlib import Path\n\nimport numpy as np\n\nimport pandas as pd\n\nimport tensorflow as tf\n\nimport tf_keras as keras\n\nfrom sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,\n    roc_auc_score, average_precision_score, confusion_matrix, balanced_accuracy_score)\n\nREGISTRY = {\n    'vgg16': ('teacher','VGG16','vgg16'),\n    'resnet50': ('teacher','ResNet50','resnet'),\n    'inception_v3': ('teacher','InceptionV3','inception_v3'),\n    'regnet_y_8gf': ('teacher','RegNetY080',None),\n    'densenet121': ('student','DenseNet121','densenet'),\n    'mobilenet_v2': ('student','MobileNetV2','mobilenet_v2'),\n    'mobilenet_v3': ('student', 'MobileNetV3Large', None),\n    'nasnet_mobile': ('student','NASNetMobile','nasnet'),\n}\n\ndef json_safe(value):\n    if isinstance(value, dict): return {str(k):json_safe(v) for k,v in value.items()}\n    if isinstance(value, (tuple,list)): return [json_safe(v) for v in value]\n    if isinstance(value, np.generic): return json_safe(value.item())\n    if isinstance(value, float) and not np.isfinite(value): return None\n    return value\n\ndef save_json(path, obj):\n    path=Path(path); tmp=path.with_suffix(path.suffix+'.tmp')\n    tmp.write_text(json.dumps(json_safe(obj),indent=2),encoding='utf-8'); os.replace(tmp,path)\n\ndef file_sha(path):\n    h=hashlib.sha256()\n    with open(path,'rb') as f:\n        for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)\n    return h.hexdigest()\n\ndef metrics(y, p):\n    y=np.asarray(y,dtype=int); p=np.asarray(p,dtype=float).reshape(-1)\n    if len(y)!=len(p) or not np.isfinite(p).all() or ((p<0)|(p>1)).any():\n        raise ValueError('Invalid predictions')\n    pred=(p>=.5).astype(int)\n    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()\n    return dict(precision=precision_score(y,pred,zero_division=0),\n        recall=recall_score(y,pred,zero_division=0), f1=f1_score(y,pred,zero_division=0),\n        macro_f1=f1_score(y,pred,labels=[0,1],average='macro',zero_division=0),\n        accuracy=accuracy_score(y,pred), balanced_accuracy=balanced_accuracy_score(y,pred),\n        specificity=tn/(tn+fp) if tn+fp else np.nan,\n        roc_auc=roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan,\n        average_precision=average_precision_score(y,p) if len(np.unique(y))==2 else np.nan,\n        tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp),threshold=.5,images=len(y))\n\ndef make_dataset(frame, image_dir, key, batch, seed, epoch=0, training=False, size=224):\n    paths=[str(image_dir/p) for p in frame.cache_member]\n    labels=frame.label.to_numpy(dtype=np.float32)\n    d=tf.data.Dataset.from_tensor_slices((paths,labels))\n    if training:\n        d=d.shuffle(len(frame),seed=seed+epoch,reshuffle_each_iteration=False)\n    preprocess_module=REGISTRY[key][2]\n    def load(index, pair):\n        path,label=pair\n        x=tf.image.decode_png(tf.io.read_file(path),channels=3)\n        x=tf.ensure_shape(tf.cast(x,tf.float32),[size,size,3])\n        if training:\n            # Stateless horizontal flip: image identity is not a laterality target.\n            x=tf.image.stateless_random_flip_left_right(x,seed=tf.stack([\n                tf.cast(seed+epoch*1009,tf.int32),tf.cast(index,tf.int32)]))\n        if preprocess_module:\n            x=getattr(keras.applications,preprocess_module).preprocess_input(x)\n        # MobileNetV3 / RegNet contain their own rescaling and consume [0,255].\n        return x,tf.reshape(label,[1])\n    options=tf.data.Options(); options.experimental_deterministic=True\n    options.threading.private_threadpool_size=2\n    return d.enumerate().map(load,num_parallel_calls=2,deterministic=True).batch(batch).with_options(options).prefetch(1)\n\ndef load_ready_dataset(ready_path, local_root):\n    ready_path=Path(ready_path); meta=json.loads(ready_path.read_text()); folder=ready_path.parent\n    for filename,digest in meta['file_sha256'].items():\n        if file_sha(folder/filename)!=digest: raise RuntimeError('B02 cache changed: '+filename)\n    frames={s:pd.read_csv(folder/(s+'.csv'),dtype={'patient_id':str,'sample_id':str})\n            for s in ['train','validation','test']}\n    all_d=pd.concat(frames.values(),ignore_index=True)\n    assert not all_d.sample_id.duplicated().any()\n    assert all_d.groupby('patient_id').split.nunique().max()==1\n    assert all(set(d.label)=={0,1} for d in frames.values())\n    image_dir=Path(local_root)/meta['dataset']/meta['file_sha256']['images.zip'][:16]\n    image_dir.mkdir(parents=True,exist_ok=True)\n    marker=image_dir/'EXTRACTED.json'\n    expected=meta['file_sha256']['images.zip']\n    valid=marker.exists() and json.loads(marker.read_text()).get('sha256')==expected\n    if not valid or not all((image_dir/f).is_file() for f in all_d.cache_member):\n        local_zip=image_dir/'images.zip'\n        shutil.copyfile(folder/'images.zip',local_zip)\n        if file_sha(local_zip)!=expected: raise RuntimeError('Local cache copy corrupted')\n        with zipfile.ZipFile(local_zip) as z:\n            for member in all_d.cache_member:\n                if Path(member).name!=member: raise ValueError('Unsafe cache member')\n                (image_dir/member).write_bytes(z.read(member))\n        save_json(marker,{'sha256':expected})\n    # Audit the actual extracted model inputs, including reused local cache.\n    from PIL import Image\n    for row in all_d.itertuples():\n        with Image.open(image_dir/row.cache_member) as im:\n            a=np.asarray(im)\n            if a.shape!=(meta['image_size'],meta['image_size']): raise ValueError('Unexpected image size')\n            if hashlib.sha256(a.tobytes()).hexdigest()!=row.processed_pixel_sha256:\n                raise RuntimeError('Corrupted local image cache')\n    return frames,image_dir,meta\n\ndef build_model(key, size=224, seed=42, weights='imagenet', learning_rate=1e-5, stage='finetune'):\n    keras.utils.set_random_seed(seed)\n    _, constructor, _ = REGISTRY[key]\n    base = getattr(keras.applications, constructor)(include_top=False, weights=weights,\n        input_shape=(size,size,3), pooling='avg')\n    base.trainable = stage == 'finetune'\n    for layer in base.layers:\n        if isinstance(layer, keras.layers.BatchNormalization):\n            layer.trainable = False\n            layer.fused = False\n    inp = keras.Input((size,size,3), name='preprocessed_image')\n    x = base(inp, training=False)\n    x = keras.layers.Dropout(.2, seed=seed, name='head_dropout')(x)\n    output = keras.layers.Dense(1, activation='sigmoid', dtype='float32', name='probability')(x)\n    model = keras.Model(inp, output, name=key)\n    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),\n        loss=keras.losses.BinaryCrossentropy())\n    model.optimizer.build(model.trainable_variables)\n    return model\n\ndef weight_snapshot(model):\n    # Explicitly exclude optimizer: head and fine-tuning have different variable sets.\n    base = next(layer for layer in model.layers if isinstance(layer, keras.Model))\n    return tf.train.Checkpoint(backbone=base, classifier=model.get_layer('probability'))\n\ndef probability_diagnostics(y, p):\n    p=np.asarray(p,dtype=np.float64).reshape(-1)\n    q=np.clip(p,1e-7,1-1e-7); y=np.asarray(y,dtype=np.float64)\n    return dict(probability_min=float(p.min()), probability_max=float(p.max()),\n        probability_std=float(p.std()), positive_prediction_fraction=float((p>=.5).mean()),\n        validation_bce_unweighted=float(-(y*np.log(q)+(1-y)*np.log(1-q)).mean()))\n\ndef preprocessing_audit(frames, image_dir, meta, key, protocol):\n    from PIL import Image\n    subset=frames['train'].iloc[:protocol['batch_size']]\n    raw=np.stack([np.asarray(Image.open(image_dir/name).convert('RGB'),dtype=np.float32)\n                  for name in subset.cache_member])\n    ds=make_dataset(subset,image_dir,key,protocol['batch_size'],protocol['training_seed'],size=meta['image_size'])\n    x,y=next(iter(ds)); actual=x.numpy()\n    if key in ['vgg16','resnet50']:\n        expected=raw[...,::-1]-np.array([103.939,116.779,123.68],dtype=np.float32)\n    elif key=='densenet121':\n        expected=(raw/255.-np.array([.485,.456,.406],dtype=np.float32))/np.array([.229,.224,.225],dtype=np.float32)\n    elif key in ['inception_v3','mobilenet_v2','nasnet_mobile']:\n        expected=raw/127.5-1.\n    else: expected=raw  # RegNet and MobileNetV3 use internal preprocessing.\n    np.testing.assert_allclose(actual,expected,rtol=1e-6,atol=2e-5)\n    np.testing.assert_equal(y.numpy().reshape(-1),subset.label.to_numpy())\n    return dict(model=key,constructor=REGISTRY[key][1],preprocessing=REGISTRY[key][2],\n        checked_split='train',checked_images=len(subset),input_min=float(actual.min()),\n        input_max=float(actual.max()),input_std=float(actual.std()),formula_check='passed')\n\ndef environment_record():\n    return dict(python=platform.python_version(),tensorflow=tf.__version__,tf_keras=keras.__version__,\n        gpu=[tf.config.experimental.get_device_details(g) for g in tf.config.list_physical_devices('GPU')])\n"
KD_ENGINE_SOURCE = '# B05 Hinton KD — binary-logit adaptation; model weights remain B04 compatible.\nimport warnings\nfrom sklearn.metrics import ConfusionMatrixDisplay\n\nTEACHER_KEYS = (\'vgg16\', \'densenet121\')\nSTUDENT_KEYS = (\'mobilenet_v2\', \'nasnet_mobile\')\n\ndef object_sha(value):\n    return hashlib.sha256(json.dumps(json_safe(value), sort_keys=True).encode()).hexdigest()\n\ndef read_json(path):\n    return json.loads(Path(path).read_text())\n\ndef guarded_json(path, value):\n    if Path(path).exists() and read_json(path) != json_safe(value):\n        raise RuntimeError(\'Existing artifact has a different configuration: \'+str(path))\n    save_json(path, value)\n\ndef check_frame(frame, reference):\n    columns=[\'sample_id\',\'patient_id\',\'label\']\n    if frame.sample_id.duplicated().any() or set(frame.sample_id)!=set(reference.sample_id):\n        raise RuntimeError(\'Prediction IDs differ from the frozen cohort\')\n    aligned=frame.set_index(\'sample_id\').loc[reference.sample_id].reset_index()\n    for col in columns:\n        if not np.array_equal(aligned[col].to_numpy(), reference[col].to_numpy()):\n            raise RuntimeError(\'Prediction identity/label mismatch: \'+col)\n    return aligned\n\ndef source_run(root, dataset, key, seed, meta, protocol_folder=None):\n    if protocol_folder:\n        candidates=[Path(root)/dataset/protocol_folder/key/f\'seed_{seed}\']\n    else:\n        candidates=list((Path(root)/dataset).glob(f\'*/{key}/seed_{seed}\'))\n    valid=[]\n    for folder in candidates:\n        if not (folder/\'COMPLETE.json\').is_file() or not (folder/\'config.json\').is_file():\n            continue\n        cfg=read_json(folder/\'config.json\')\n        if (cfg.get(\'model\')==key and cfg.get(\'dataset\')==dataset and\n            cfg.get(\'training_seed\')==seed and cfg.get(\'variant\')==\'full_512\' and\n            cfg.get(\'cohort_sha256\')==meta[\'cohort_sha256\'] and\n            cfg.get(\'images_sha256\')==meta[\'file_sha256\'][\'images.zip\']):\n            valid.append(folder)\n    if len(valid)!=1:\n        raise RuntimeError(f\'Need exactly one completed B04B run for {dataset}/{key}/seed_{seed}; found {valid}. Check B04_PROTOCOL_FOLDERS.\')\n    folder=valid[0];cfg=read_json(folder/\'config.json\');done=read_json(folder/\'COMPLETE.json\')\n    if done.get(\'config_sha256\')!=file_sha(folder/\'config.json\'):\n        raise RuntimeError(\'B04B config changed after completion: \'+str(folder))\n    checks={\'image_size\':512,\'threshold\':.5,\'head\':\'GAP_dropout0.2_sigmoid\',\n            \'batch_norm\':\'frozen\',\'batch_norm_implementation\':\'non_fused\'}\n    for k,v in checks.items():\n        if cfg.get(k)!=v:raise RuntimeError(f\'Unexpected B04B {k}: {cfg.get(k)}; expected {v}\')\n    if cfg.get(\'data_file_sha256\')!=meta[\'file_sha256\']:\n        raise RuntimeError(\'B04B split/cache hashes differ from current READY.json\')\n    prefix=Path(done[\'best_checkpoint\'])\n    if not Path(str(prefix)+\'.index\').exists():\n        prefix=folder/done[\'best_stage\']/\'best\'/prefix.name\n    checkpoint_files=sorted(prefix.parent.glob(prefix.name+\'.*\'))\n    if not Path(str(prefix)+\'.index\').exists() or not any(\'.data-\' in p.name for p in checkpoint_files):\n        raise FileNotFoundError(\'Missing teacher weights: \'+str(prefix))\n    return dict(folder=str(folder),checkpoint=str(prefix),config_sha256=file_sha(folder/\'config.json\'),\n        checkpoint_sha256={p.name:file_sha(p) for p in checkpoint_files},\n        model=key,teacher_seed=seed,best_stage=done[\'best_stage\'],\n        validation_predictions_sha256=file_sha(folder/\'validation_predictions.csv\'),\n        b04_tensorflow=cfg.get(\'tf_version\'),b04_tf_keras=cfg.get(\'tf_keras_version\'))\n\ndef logits_view(model):\n    # Compute pre-sigmoid z directly. Never recover logit from clipped probabilities.\n    classifier=model.get_layer(\'probability\')\n    features=keras.Model(model.input,classifier.input)\n    class View(keras.Model):\n        def call(self,x,training=False):\n            f=features(x,training=training)\n            return tf.linalg.matmul(f,classifier.kernel)+classifier.bias\n    return View()\n\ndef raw_dataset(frame,image_dir,batch,size,seed,epoch=0,training=False,teacher_logits=None,fixed_flip=False):\n    paths=[str(image_dir/p) for p in frame.cache_member]\n    labels=frame.label.to_numpy(np.float32)\n    if teacher_logits is None:teacher_logits=np.zeros((len(frame),2),np.float32)\n    teacher_logits=np.asarray(teacher_logits,np.float32)\n    if teacher_logits.shape!=(len(frame),2) or not np.isfinite(teacher_logits).all():\n        raise ValueError(\'Invalid aligned teacher logits\')\n    d=tf.data.Dataset.from_tensor_slices((paths,labels,teacher_logits))\n    if training:d=d.shuffle(len(frame),seed=seed+epoch,reshuffle_each_iteration=False)\n    def load(index,values):\n        path,label,z=values\n        x=tf.ensure_shape(tf.cast(tf.image.decode_png(tf.io.read_file(path),channels=3),tf.float32),[size,size,3])\n        flip=tf.constant(fixed_flip)\n        if training:\n            flip=tf.random.stateless_uniform([],seed=tf.stack([tf.cast(seed+epoch*1009,tf.int32),tf.cast(index,tf.int32)]))<.5\n        x=tf.cond(flip,lambda:tf.image.flip_left_right(x),lambda:x)\n        teacher_z=tf.where(flip,z[1],z[0])\n        return x,tf.reshape(label,[1]),tf.reshape(teacher_z,[1])\n    options=tf.data.Options();options.experimental_deterministic=True\n    options.threading.private_threadpool_size=2\n    return d.enumerate().map(load,num_parallel_calls=2,deterministic=True).batch(batch).with_options(options).prefetch(1)\n\ndef preprocess(x,key):\n    return getattr(keras.applications,REGISTRY[key][2]).preprocess_input(x)\n\ndef logit_input_dataset(frame,image_dir,key,batch,size,seed,flip=False):\n    # Same preprocessing and ordering as B04; only add the chosen deterministic view.\n    ds=raw_dataset(frame,image_dir,batch,size,seed,fixed_flip=flip)\n    @tf.autograph.experimental.do_not_convert\n    def inputs(x,y,z):\n        return preprocess(x,key)\n    return ds.map(inputs,num_parallel_calls=1,deterministic=True)\n\ndef predict_logits(view,frame,image_dir,key,batch,size,seed,flip=False):\n    ds=logit_input_dataset(frame,image_dir,key,batch,size,seed,flip)\n    # Match B04 graph-mode predict, not a Python eager loop over batches.\n    a=np.asarray(view.predict(ds,verbose=0),dtype=np.float32).reshape(-1)\n    if len(a)!=len(frame) or not np.isfinite(a).all():raise ValueError(\'Invalid model logits\')\n    return a\n\ndef comparison_stats(reference,current):\n    reference=np.asarray(reference,dtype=np.float64).reshape(-1)\n    current=np.asarray(current,dtype=np.float64).reshape(-1)\n    if len(reference)!=len(current) or not np.isfinite(reference).all() or not np.isfinite(current).all():\n        raise ValueError(\'Invalid audit arrays\')\n    delta=np.abs(reference-current)\n    return dict(max_abs_difference=float(delta.max()),mean_abs_difference=float(delta.mean()),\n        p99_abs_difference=float(np.quantile(delta,.99)),\n        threshold_prediction_disagreements=int(np.sum((reference>=.5)!=(current>=.5))),\n        outside_tolerance=int(np.sum(~np.isclose(current,reference,rtol=1e-4,atol=2e-5))),\n        passed=bool(np.allclose(current,reference,rtol=1e-4,atol=2e-5)))\n\ndef write_teacher_failure(out,audit,table,source):\n    audit=dict(audit,passed=False)\n    save_json(out/\'teacher_audit.json\',audit)\n    table.to_csv(out/\'audit_comparison.csv\',index=False)\n    save_json(out/\'runtime.json\',environment_record())\n    report=out/\'teacher_audit_failure.zip\'\n    with zipfile.ZipFile(report,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for name in [\'manifest.json\',\'teacher_audit.json\',\'audit_comparison.csv\',\'runtime.json\']:\n            z.write(out/name,name)\n        for name in [\'config.json\',\'COMPLETE.json\',\'environment.json\',\'validation_metrics.json\']:\n            p=Path(source[\'folder\'])/name\n            if p.exists():z.write(p,\'b04_source/\'+name)\n    globals()[\'LAST_TEACHER_AUDIT_FAILURE\']=str(report)\n    print(\'AUDIT FAILED:\',audit[\'failure_stage\'],flush=True)\n    print(\'Diagnostic report:\',report,flush=True)\n    print(table.sort_values(\'abs_diff_saved_vs_native\',ascending=False).head(10).to_string(index=False),flush=True)\n    raise RuntimeError(f"Teacher audit failed at {audit[\'failure_stage\']}. No cache was committed. Send diagnostic ZIP: {report}")\n\ndef build_teacher_cache(source,frames,image_dir,meta,cache_root,batch,seed):\n    globals()[\'LAST_TEACHER_AUDIT_FAILURE\']=None\n    cfg=read_json(Path(source[\'folder\'])/\'config.json\')\n    audit_batch=int(cfg[\'batch_size\'])\n    if audit_batch<=0:raise ValueError(\'Invalid B04 reference batch size\')\n    tf32=bool(tf.config.experimental.tensor_float_32_execution_enabled())\n    manifest=dict(source=source,cohort_sha256=meta[\'cohort_sha256\'],data_sha256=meta[\'file_sha256\'],\n        format=\'raw_binary_logits_original_and_horizontal_flip_graph_predict_v2\',\n        inference_batch_size=int(batch),audit_batch_size=audit_batch,tf32_enabled=tf32,\n        tensorflow=tf.__version__,tf_keras=keras.__version__,builder_sha256=ENGINE_SHA256)\n    out=Path(cache_root)/meta[\'dataset\']/source[\'model\']/object_sha(manifest)[:16]\n    out.mkdir(parents=True,exist_ok=True);guarded_json(out/\'manifest.json\',manifest)\n    cache_file=out/\'train_logits.csv\';done=out/\'COMPLETE.json\'\n    if done.exists():\n        if file_sha(cache_file)!=read_json(done)[\'csv_sha256\']:raise RuntimeError(\'Teacher cache changed\')\n        if not read_json(out/\'teacher_audit.json\').get(\'passed\'):raise RuntimeError(\'Cache lacks a passing teacher audit\')\n        table=check_frame(pd.read_csv(cache_file,dtype={\'sample_id\':str,\'patient_id\':str},float_precision=\'round_trip\'),frames[\'train\'])\n        return table[[\'logit_original\',\'logit_flipped\']].to_numpy(np.float32),out\n    keras.backend.clear_session();gc.collect()\n    model=build_model(source[\'model\'],meta[\'image_size\'],source[\'teacher_seed\'],weights=None,stage=\'head\')\n    weight_snapshot(model).restore(source[\'checkpoint\']).assert_existing_objects_matched()\n    model.trainable=False;view=logits_view(model)\n    start=time.perf_counter();val=frames[\'validation\']\n    saved=check_frame(pd.read_csv(Path(source[\'folder\'])/\'validation_predictions.csv\',\n        dtype={\'sample_id\':str,\'patient_id\':str},float_precision=\'round_trip\'),val)\n    # Stage 1: exactly the original probability interface and original B04 batch size.\n    native_ds=make_dataset(val,image_dir,source[\'model\'],audit_batch,source[\'teacher_seed\'],size=meta[\'image_size\'])\n    p=np.asarray(model.predict(native_ds,verbose=0),dtype=np.float32).reshape(-1)\n    original=saved.probability.to_numpy(np.float64)\n    audit=dict(passed=False,patch=\'teacher_native_graph_audit_v2\',source=source,\n        audit_batch_size=audit_batch,cache_batch_size=int(batch),tf32_enabled=tf32,\n        current_runtime=environment_record(),reference_tf=cfg.get(\'tf_version\'),reference_tf_keras=cfg.get(\'tf_keras_version\'),\n        tolerances=dict(rtol=1e-4,atol=2e-5),native_vs_saved=comparison_stats(original,p),\n        current_validation_metrics=metrics(val.label,p))\n    comparison=val[[\'sample_id\',\'patient_id\',\'label\']].copy()\n    comparison[\'saved_probability\']=original;comparison[\'native_probability\']=p\n    comparison[\'abs_diff_saved_vs_native\']=np.abs(original-p)\n    if not audit[\'native_vs_saved\'][\'passed\']:\n        audit[\'failure_stage\']=\'native_predict_vs_B04_saved\'\n        write_teacher_failure(out,audit,comparison,source)\n    # Stage 2: separately verify the logit view against the original sigmoid model.\n    val_z=predict_logits(view,val,image_dir,source[\'model\'],audit_batch,meta[\'image_size\'],seed)\n    p_logit=tf.sigmoid(val_z).numpy()\n    audit[\'logit_vs_native\']=comparison_stats(p,p_logit)\n    comparison[\'logit_probability\']=p_logit\n    if not audit[\'logit_vs_native\'][\'passed\']:\n        audit[\'failure_stage\']=\'logit_view_vs_native_predict\'\n        write_teacher_failure(out,audit,comparison,source)\n    # Explicitly check numerical drift if inference cache batches differ from B04.\n    if int(batch)!=audit_batch:\n        val_z_cache=predict_logits(view,val,image_dir,source[\'model\'],batch,meta[\'image_size\'],seed)\n        comparison[\'cache_batch_probability\']=tf.sigmoid(val_z_cache).numpy()\n        audit[\'cache_batch_vs_reference\']=comparison_stats(p,comparison[\'cache_batch_probability\'])\n        if not audit[\'cache_batch_vs_reference\'][\'passed\']:\n            audit[\'failure_stage\']=\'cache_batch_vs_B04_batch; use TEACHER_BATCH_SIZE=\'+str(audit_batch)\n            write_teacher_failure(out,audit,comparison,source)\n    audit.update(passed=True,max_abs_probability_difference=audit[\'native_vs_saved\'][\'max_abs_difference\'],\n        threshold_prediction_disagreements=audit[\'native_vs_saved\'][\'threshold_prediction_disagreements\'])\n    save_json(out/\'teacher_audit.json\',audit)\n    comparison.to_csv(out/\'audit_comparison.csv\',index=False)\n    print(\'Teacher audit passed:\',source[\'model\'],\'native max diff=\',audit[\'native_vs_saved\'][\'max_abs_difference\'],\n        \'logit max diff=\',audit[\'logit_vs_native\'][\'max_abs_difference\'],flush=True)\n    # Additional check of cache inference on train and on BOTH deterministic views.\n    probe=frames[\'train\'].iloc[:min(len(frames[\'train\']),2*batch+1)]\n    for flip in [False,True]:\n        ds=logit_input_dataset(probe,image_dir,source[\'model\'],batch,meta[\'image_size\'],seed,flip)\n        probe_p=np.asarray(model.predict(ds,verbose=0)).reshape(-1)\n        probe_z=predict_logits(view,probe,image_dir,source[\'model\'],batch,meta[\'image_size\'],seed,flip)\n        stat=comparison_stats(probe_p,tf.sigmoid(probe_z).numpy())\n        audit[\'train_flipped_probe\' if flip else \'train_original_probe\']=stat\n        if not stat[\'passed\']:\n            audit[\'failure_stage\']=\'train_view_logit_check_flip=\'+str(flip)\n            write_teacher_failure(out,audit,comparison,source)\n    values=[]\n    for flip in [False,True]:\n        print(\'Caching teacher\',source[\'model\'],\'horizontal_flip=\',flip,flush=True)\n        values.append(predict_logits(view,frames[\'train\'],image_dir,source[\'model\'],batch,meta[\'image_size\'],seed,flip))\n    table=frames[\'train\'][[\'sample_id\',\'patient_id\',\'label\']].copy()\n    table[\'logit_original\'],table[\'logit_flipped\']=values\n    tmp=out/\'train_logits.tmp.csv\';table.to_csv(tmp,index=False);os.replace(tmp,cache_file)\n    save_json(out/\'teacher_audit.json\',audit)\n    save_json(done,dict(csv_sha256=file_sha(cache_file),cache_minutes=(time.perf_counter()-start)/60,\n        images=len(table),views=2,teacher_trainable=False))\n    del model,view;keras.backend.clear_session();gc.collect()\n    return table[[\'logit_original\',\'logit_flipped\']].to_numpy(np.float32),out\n\ndef loss_terms(y,student_z,teacher_z,class_weights,alpha,temperature):\n    y=tf.cast(y,tf.float32);student_z=tf.cast(student_z,tf.float32)\n    teacher_z=tf.stop_gradient(tf.cast(teacher_z,tf.float32))\n    hard=tf.nn.sigmoid_cross_entropy_with_logits(labels=y,logits=student_z)\n    weights=tf.where(y>.5,tf.cast(class_weights[1],tf.float32),tf.cast(class_weights[0],tf.float32))\n    hard=tf.reduce_mean(weights*hard)\n    # Two-class Bernoulli KL: q=[1-sigmoid(z_t/T),sigmoid(z_t/T)].\n    # softmax over a single output would always be 1 and is invalid.\n    a=teacher_z/temperature;b=student_z/temperature;q=tf.sigmoid(a)\n    kl=q*(tf.math.log_sigmoid(a)-tf.math.log_sigmoid(b))+(1-q)*(tf.math.log_sigmoid(-a)-tf.math.log_sigmoid(-b))\n    soft=tf.reduce_mean(kl)*temperature**2\n    return (1-alpha)*hard+alpha*soft,hard,soft\n\ndef weights_sha(model):\n    h=hashlib.sha256()\n    for w in model.weights:h.update(w.numpy().tobytes())\n    return h.hexdigest()\n\ndef export_result(model,key,frame,image_dir,meta,folder,split,protocol,extra):\n    ds=make_dataset(frame,image_dir,key,protocol[\'batch_size\'],protocol[\'training_seed\'],size=meta[\'image_size\'])\n    p=model.predict(ds,verbose=0).reshape(-1)\n    result=dict(metrics(frame.label,p),model=key,dataset=meta[\'dataset\'],training_seed=protocol[\'training_seed\'],\n        split=split,role=\'student\',**extra)\n    table=frame[[\'sample_id\',\'patient_id\',\'label\']].copy()\n    table[\'probability\']=p.astype(np.float64);table[\'prediction\']=(p>=.5).astype(int)\n    for k in [\'model\',\'dataset\',\'training_seed\',\'split\',\'teacher\',\'method\']:table[k]=result[k]\n    table.to_csv(folder/f\'{split}_predictions.csv\',index=False);save_json(folder/f\'{split}_metrics.json\',result)\n    import matplotlib.pyplot as plt\n    names=[\'BI-RADS 1/2\',\'BI-RADS 5/6\'] if meta[\'dataset\']==\'inbreast\' else [\'Benign\',\'Malignant\']\n    fig,ax=plt.subplots(figsize=(5.6,5))\n    ConfusionMatrixDisplay(np.array([[result[\'tn\'],result[\'fp\']],[result[\'fn\'],result[\'tp\']]]),display_labels=names).plot(\n        ax=ax,cmap=\'Blues\',values_format=\'d\',im_kw={\'vmin\':0,\'vmax\':int(frame.label.value_counts().max())})\n    ax.set_title(f"{key} | {result[\'method\']} | {result[\'teacher\']}\\n{meta[\'dataset\']} {split}; seed {protocol[\'training_seed\']}; threshold 0.5",fontsize=9)\n    fig.tight_layout();fig.savefig(folder/f\'{split}_confusion_matrix.png\',dpi=180);plt.close(fig)\n    return result\n\ndef run_phase(key,teacher,teacher_values,frames,image_dir,meta,out,protocol,stage,initial_checkpoint=None):\n    out.mkdir(parents=True,exist_ok=True)\n    if (out/\'COMPLETE.json\').exists():return read_json(out/\'validation_metrics.json\'),read_json(out/\'COMPLETE.json\')\n    keras.backend.clear_session();gc.collect()\n    seed=protocol[\'training_seed\'];head=stage==\'head\';latest=tf.train.latest_checkpoint(str(out/\'last\'))\n    lr=protocol[\'head_learning_rate\'] if head else protocol[\'finetune_learning_rate\']\n    model=build_model(key,meta[\'image_size\'],seed,weights=\'imagenet\' if head and not latest else None,learning_rate=lr,stage=stage)\n    snapshot=weight_snapshot(model)\n    if head and not latest:\n        guarded_json(out/\'initialization.json\',dict(weights_sha256=weights_sha(model),seed=seed,weights=\'imagenet\'))\n        if teacher!=\'none\':\n            control=out.parents[2]/(key+\'__from__none\')/f\'seed_{seed}\'/\'head\'/\'initialization.json\'\n            if not control.exists() or read_json(control)!=read_json(out/\'initialization.json\'):\n                raise RuntimeError(\'KD initial weights differ from matched supervised control\')\n    if not head and not latest:\n        if not initial_checkpoint:raise RuntimeError(\'Fine-tune requires best head weights\')\n        snapshot.restore(initial_checkpoint).assert_existing_objects_matched()\n    view=logits_view(model);optimizer=model.optimizer\n    epoch_v=tf.Variable(0,dtype=tf.int64,trainable=False)\n    best_f=tf.Variable(-1.,dtype=tf.float64,trainable=False);best_r=tf.Variable(-1.,dtype=tf.float64,trainable=False)\n    wait=tf.Variable(0,dtype=tf.int64,trainable=False);best_epoch=tf.Variable(0,dtype=tf.int64,trainable=False)\n    best_path=tf.Variable(\'\',dtype=tf.string,trainable=False);hist_v=tf.Variable(\'[]\',dtype=tf.string,trainable=False)\n    checkpoint=tf.train.Checkpoint(model=model,optimizer=optimizer,epoch=epoch_v,best_f1=best_f,best_recall=best_r,\n        wait=wait,best_epoch=best_epoch,best_path=best_path,history=hist_v)\n    manager=tf.train.CheckpointManager(checkpoint,str(out/\'last\'),max_to_keep=2)\n    best_manager=tf.train.CheckpointManager(snapshot,str(out/\'best\'),max_to_keep=2)\n    if latest:checkpoint.restore(latest).assert_existing_objects_matched()\n    history=json.loads(hist_v.numpy().decode())\n    train=frames[\'train\'];val=frames[\'validation\'];counts=train.label.value_counts()\n    cw={k:len(train)/(2*int(counts[k])) for k in [0,1]}\n    alpha=protocol[\'alpha\'] if teacher!=\'none\' else 0.;temperature=protocol[\'temperature\']\n    @tf.function(reduce_retracing=True)\n    def train_step(x,y,z):\n        with tf.GradientTape() as tape:\n            student_z=view(preprocess(x,key),training=True)\n            total,hard,soft=loss_terms(y,student_z,z,cw,alpha,temperature)\n            tf.debugging.assert_all_finite(total,\'Nonfinite training loss\')\n        variables=model.trainable_variables;grad=tape.gradient(total,variables)\n        if any(g is None for g in grad):raise RuntimeError(\'Missing Student gradient\')\n        for g in grad:tf.debugging.assert_all_finite(g,\'Nonfinite Student gradient\')\n        optimizer.apply_gradients(zip(grad,variables))\n        return total,hard,soft\n    budget=protocol[\'head_epochs\'] if head else protocol[\'finetune_epochs\']\n    val_ds=make_dataset(val,image_dir,key,protocol[\'batch_size\'],seed,size=meta[\'image_size\'])\n    for epoch in range(int(epoch_v.numpy()),budget):\n        if not head and protocol.get(\'early_stopping\',True) and int(wait.numpy())>=protocol[\'patience\']:break\n        global_epoch=epoch if head else protocol[\'head_epochs\']+epoch\n        keras.utils.set_random_seed(seed+global_epoch*1009)\n        ds=raw_dataset(train,image_dir,protocol[\'batch_size\'],meta[\'image_size\'],seed,global_epoch,True,teacher_values)\n        start=time.perf_counter();sums=np.zeros(3);seen=0\n        for x,y,z in ds:\n            losses=train_step(x,y,z);n=int(x.shape[0]);sums+=np.array([float(v.numpy()) for v in losses])*n;seen+=n\n        elapsed=time.perf_counter()-start\n        p=model.predict(val_ds,verbose=0).reshape(-1);score=metrics(val.label,p)\n        row=dict(phase_epoch=epoch+1,global_epoch=global_epoch+1,stage=stage,loss=sums[0]/seen,\n            supervised_loss=sums[1]/seen,distillation_loss_T2=sums[2]/seen if alpha else 0.,\n            train_seconds=elapsed,learning_rate=lr,**score)\n        improved=(score[\'macro_f1\'],score[\'recall\'])>(float(best_f.numpy()),float(best_r.numpy()))\n        if improved:\n            best_f.assign(score[\'macro_f1\']);best_r.assign(score[\'recall\']);wait.assign(0);best_epoch.assign(epoch+1)\n            best_path.assign(best_manager.save(checkpoint_number=epoch+1))\n        else:wait.assign_add(1)\n        history.append(row);epoch_v.assign(epoch+1);hist_v.assign(json.dumps(json_safe(history)))\n        manager.save(checkpoint_number=epoch+1)\n        pd.DataFrame(history).to_csv(out/\'history.csv\',index=False)\n        print(f\'{key} <- {teacher} | {stage} {epoch+1}/{budget} | loss={row["loss"]:.4f} F1={score["f1"]:.4f} macro-F1={score["macro_f1"]:.4f}\',flush=True)\n    if not protocol.get(\'early_stopping\',True) and int(epoch_v.numpy())!=budget:\n        raise RuntimeError(\'Fixed-budget run did not complete all epochs\')\n    selected=best_path.numpy().decode()\n    if not selected:raise RuntimeError(\'No completed training epoch\')\n    snapshot.restore(selected).assert_existing_objects_matched()\n    method=\'hinton_kd\' if teacher!=\'none\' else \'supervised_control\'\n    result=export_result(model,key,val,image_dir,meta,out,\'validation\',protocol,dict(teacher=teacher,method=method,\n        alpha=alpha,temperature=temperature if alpha else None,best_stage=stage,best_phase_epoch=int(best_epoch.numpy()),\n        parameters=model.count_params(),epochs_completed=int(epoch_v.numpy()),total_train_minutes=sum(r[\'train_seconds\'] for r in history)/60))\n    state=dict(best_checkpoint=selected,last_checkpoint=manager.latest_checkpoint,stage=stage,validation_only=True)\n    save_json(out/\'COMPLETE.json\',state);pd.DataFrame(history).to_csv(out/\'history.csv\',index=False)\n    del model,view,train_step,optimizer,checkpoint,snapshot,manager,best_manager\n    keras.backend.clear_session();gc.collect()\n    return result,state\n\ndef run_experiment(key,teacher,teacher_values,frames,image_dir,meta,root,protocol,sources):\n    tag=key+\'__from__\'+teacher;out=Path(root)/tag/f"seed_{protocol[\'training_seed\']}";out.mkdir(parents=True,exist_ok=True)\n    cfg=dict(protocol,student=key,teacher=teacher,teacher_source=sources.get(teacher),method=\'hinton_kd\' if teacher!=\'none\' else \'supervised_control\')\n    guarded_json(out/\'config.json\',cfg)\n    if (out/\'COMPLETE.json\').exists():\n        if read_json(out/\'COMPLETE.json\')[\'config_sha256\']!=file_sha(out/\'config.json\'):raise RuntimeError(\'Completion/config mismatch\')\n        return read_json(out/\'validation_metrics.json\'),out\n    save_json(out/\'environment.json\',environment_record())\n    save_json(out/\'preprocessing_audit.json\',preprocessing_audit(frames,image_dir,meta,key,protocol))\n    head,hs=run_phase(key,teacher,teacher_values,frames,image_dir,meta,out/\'head\',protocol,\'head\')\n    ft,fs=run_phase(key,teacher,teacher_values,frames,image_dir,meta,out/\'finetune\',protocol,\'finetune\',hs[\'best_checkpoint\'])\n    use_ft=(ft[\'macro_f1\'],ft[\'recall\'])>(head[\'macro_f1\'],head[\'recall\'])\n    chosen,state,stage=(ft,fs,\'finetune\') if use_ft else (head,hs,\'head\')\n    result=dict(chosen,best_epoch=chosen[\'best_phase_epoch\']+(protocol[\'head_epochs\'] if use_ft else 0),\n        total_train_minutes=head[\'total_train_minutes\']+ft[\'total_train_minutes\'],\n        epochs_completed=head[\'epochs_completed\']+ft[\'epochs_completed\'],run_folder=str(out))\n    for name in [\'validation_predictions.csv\',\'validation_confusion_matrix.png\']:shutil.copyfile(out/stage/name,out/name)\n    pd.concat([pd.read_csv(out/s/\'history.csv\') for s in [\'head\',\'finetune\']],ignore_index=True).to_csv(out/\'history.csv\',index=False)\n    save_json(out/\'stage_comparison.json\',dict(head=head,finetune=ft,selected=stage))\n    save_json(out/\'validation_metrics.json\',result)\n    save_json(out/\'COMPLETE.json\',dict(best_checkpoint=state[\'best_checkpoint\'],best_stage=stage,config_sha256=file_sha(out/\'config.json\'),validation_only=True))\n    return result,out\n\ndef evaluation_lock(root,folders,meta,protocol):\n    # Must include both controls and all four pairs before opening B05 test evaluation.\n    expected={k+\'__from__\'+t for k in STUDENT_KEYS for t in (\'none\',)+TEACHER_KEYS}\n    if {p.parent.name for p in folders}!=expected:raise RuntimeError(\'Complete all 6 runs before freezing test evaluation\')\n    record=dict(protocol_sha256=object_sha(protocol),test_csv_sha256=meta[\'file_sha256\'][\'test.csv\'],\n        runs={str(p):dict(config_sha256=file_sha(p/\'config.json\'),complete=read_json(p/\'COMPLETE.json\')) for p in folders},\n        warning=\'B03 test was previously inspected; this is not a never-seen holdout.\')\n    guarded_json(Path(root)/\'TEST_LOCK.json\',record)\n    return record\n\ndef test_student(out,frames,image_dir,meta,protocol):\n    cfg=read_json(out/\'config.json\');complete=read_json(out/\'COMPLETE.json\')\n    if (out/\'test_metrics.json\').exists():return read_json(out/\'test_metrics.json\')\n    keras.backend.clear_session();gc.collect()\n    model=build_model(cfg[\'student\'],meta[\'image_size\'],protocol[\'training_seed\'],weights=None,stage=\'head\')\n    weight_snapshot(model).restore(complete[\'best_checkpoint\']).assert_existing_objects_matched()\n    result=export_result(model,cfg[\'student\'],frames[\'test\'],image_dir,meta,out,\'test\',protocol,\n        dict(teacher=cfg[\'teacher\'],method=cfg[\'method\'],run_folder=str(out)))\n    del model;keras.backend.clear_session();gc.collect()\n    return result\n\ndef compare_controls(records,root,split=\'validation\'):\n    table=pd.DataFrame(records);columns=[\'precision\',\'recall\',\'f1\',\'macro_f1\',\'accuracy\',\'balanced_accuracy\',\'specificity\',\'roc_auc\',\'average_precision\']\n    rows=[]\n    for record in records:\n        if record[\'teacher\']==\'none\':continue\n        control=table[(table.model==record[\'model\'])&(table.teacher==\'none\')]\n        if len(control)!=1:raise RuntimeError(\'One matched supervised control is required\')\n        base=control.iloc[0];row={k:record[k] for k in [\'dataset\',\'training_seed\',\'model\',\'teacher\',\'method\',\'split\']}\n        for metric in columns:row[\'delta_\'+metric]=record[metric]-base[metric]\n        rows.append(row)\n    result=pd.DataFrame(rows);result.to_csv(Path(root)/f\'{split}_kd_vs_control.csv\',index=False)\n    return result\n'

import hashlib
B04_COMPAT_SHA256=hashlib.sha256(B04_COMPAT_SOURCE.encode()).hexdigest()
ENGINE_SHA256=hashlib.sha256(KD_ENGINE_SOURCE.encode()).hexdigest()
# Register exact source text so inspect/AutoGraph can locate dynamically loaded functions.

import linecache
for source_name, source_text in [('b04_compat.py', B04_COMPAT_SOURCE), ('b05_hinton_engine.py', KD_ENGINE_SOURCE)]:
    linecache.cache[source_name]=(len(source_text),None,source_text.splitlines(True),source_name)
    exec(compile(source_text,source_name,'exec'),globals())

# Numerical check: KL is identically ~0, the Teacher gradient is cut off, and the Student receives the gradient.
y=tf.constant([[0.],[1.]])
s=tf.Variable([[-.8],[1.2]],dtype=tf.float32)
t=tf.Variable([[-1.5],[2.]],dtype=tf.float32)
with tf.GradientTape(persistent=True) as tape:
    loss,hard,soft=loss_terms(y,s,t,{0:1.,1:1.},.5,2.)
assert tape.gradient(loss,t) is None
assert bool(tf.reduce_any(tf.not_equal(tape.gradient(loss,s),0)).numpy())
_,_,same=loss_terms(y,t,t,{0:1.,1:1.},.5,2.)
np.testing.assert_allclose(float(same.numpy()),0,atol=1e-6)
np.testing.assert_allclose(float(loss_terms(y,s,t,{0:1.,1:1.},0.,2.)[0]),float(hard),atol=1e-6)
del tape,s,t
print('Loss sanity checks passed. Engine:',ENGINE_SHA256[:12])

## Cell 4 — Dataset and audit split
Reading the frozen CSV/cache B04A, checking the hash, sample IDs, group overlaps and pre-processed pixels. The test is audited for integrity only; no inference or model selection has yet been carried out.

In [ ]:
if DATASET!='inbreast' or TRAINING_SEED!=42 or TEACHER_SEED!=42:
    raise ValueError('Ablation ini khusus INbreast seed 42')
if EARLY_STOPPING is not False or HEAD_EPOCHS!=5 or FINETUNE_EPOCHS!=25:
    raise ValueError('Gunakan 5+25 epoch dan EARLY_STOPPING=False')
if EVALUATE_TEST:raise ValueError('Test tetap nonaktif selama ablation')
if STUDENTS_TO_RUN!=['mobilenet_v2','nasnet_mobile'] or TEACHERS_TO_RUN!=['vgg16','densenet121']:
    raise ValueError('Ablation harus mencakup kedua kontrol dan empat pasangan KD')
if DATASET not in ['cbis_ddsm','cmmd','inbreast','mias']:raise ValueError('Unknown dataset')
if not STUDENTS_TO_RUN or len(set(STUDENTS_TO_RUN))!=len(STUDENTS_TO_RUN) or not set(STUDENTS_TO_RUN)<=set(STUDENT_KEYS):raise ValueError('Invalid students')
if not TEACHERS_TO_RUN or len(set(TEACHERS_TO_RUN))!=len(TEACHERS_TO_RUN) or not set(TEACHERS_TO_RUN)<=set(TEACHER_KEYS):raise ValueError('Invalid teachers')
if not isinstance(TRAINING_SEED,int) or TRAINING_SEED<0 or not isinstance(TEACHER_SEED,int) or TEACHER_SEED<0:raise ValueError('Invalid seed')
if not 0<ALPHA<1 or not np.isfinite(TEMPERATURE) or TEMPERATURE<=0:raise ValueError('Invalid KD loss configuration')
if min(BATCH_SIZE,TEACHER_BATCH_SIZE,HEAD_EPOCHS,FINETUNE_EPOCHS,PATIENCE)<=0:raise ValueError('Invalid budget')
if not B04A_REGISTRY.is_file():raise FileNotFoundError(f'Check B04A_REGISTRY: {B04A_REGISTRY}')
READY_REGISTRY=read_json(B04A_REGISTRY)
entry=READY_REGISTRY.get(DATASET,{}).get('full_512',{})
if entry.get('status')!='READY':raise RuntimeError('B04A full_512 is not yet READY')
FRAMES,IMAGE_DIR,DATA_META=load_ready_dataset(Path(entry['path'])/'READY.json',LOCAL_IMAGES)
if DATA_META['dataset']!=DATASET or DATA_META['image_size']!=512:raise RuntimeError('Wrong cache dataset/size')
for split,frame in FRAMES.items():
    if frame[['sample_id','patient_id','label']].isna().any().any():raise ValueError('Missing identity/label')
    if set(frame['split'])!={split}:raise ValueError('CSV split column mismatch')
AUDIT=pd.DataFrame([dict(split=s,images=len(f),patient_groups=f.patient_id.nunique(),
    negative=int((f.label==0).sum()),positive=int((f.label==1).sum())) for s,f in FRAMES.items()])
display(AUDIT)
if DATASET=='inbreast':print('NOTE: Patient grouping follows anonymized patient IDs documented in the INbreast README. The audited B02/B04 splits have no patient-ID overlap. Target: BI-RADS 1/2 vs 5/6; not pathology-confirmed labels.')
if DATASET=='mias':print('NOTE: MIAS is ‘abnormal-only’; the split size is small. A change in a single sample can significantly alter the metrics.')
if DATASET=='cmmd':print('NOTE: Report the macro-F1 and specificity alongside the F1 for positive cases due to class imbalance.')

## Cell 5 — Muat Teacher dan bangun cache ablation
Checkpoint B04B dimuat tanpa melatih ulang Teacher. Audit prediksi tetap dijalankan dengan toleransi semula. Cache baru tersimpan di folder B05B agar semua artefak B05 awal tetap utuh.


In [ ]:
SOURCES={};TEACHER_VALUES={};CACHE_FOLDERS={}
for key in TEACHERS_TO_RUN:
    source=source_run(B04_RUN_ROOT,DATASET,key,TEACHER_SEED,DATA_META,B04_PROTOCOL_FOLDERS.get(DATASET))
    SOURCES[key]=source
    try:
        values,cache_folder=build_teacher_cache(source,FRAMES,IMAGE_DIR,DATA_META,CACHE_ROOT,TEACHER_BATCH_SIZE,TRAINING_SEED)
    except RuntimeError:
        diagnostic=globals().get('LAST_TEACHER_AUDIT_FAILURE')
        if diagnostic:
            print('Check this diagnosis ZIP. Tolerances have not yet been relaxed:',diagnostic)
            from google.colab import files
            files.download(diagnostic)
        raise
    TEACHER_VALUES[key]=values;CACHE_FOLDERS[key]=cache_folder
    print(key,'ready; logits shape:',values.shape,'checkpoint:',source['checkpoint'])
display(pd.DataFrame([dict(teacher=k,**read_json(p/'teacher_audit.json')['current_validation_metrics']) for k,p in CACHE_FOLDERS.items()]).round(4))

## Cell 6 — Catat protokol ablation dan enam run
Protokol mencatat `early_stopping=False` dan hash trainer baru. Ini pencatatan konfigurasi eksperimen, bukan pembekuan untuk evaluasi test.


In [ ]:
PROTOCOL=dict(version='b05b_inbreast_fixed25_v1',early_stopping=EARLY_STOPPING,ablation='disable_early_stopping_only',dataset=DATASET,training_seed=TRAINING_SEED,teacher_seed=TEACHER_SEED,
    variant='full_512',image_size=512,split_seed=DATA_META['split_seed'],
    cohort_sha256=DATA_META['cohort_sha256'],data_sha256=DATA_META['file_sha256'],
    preprocessing_source_sha256=DATA_META['preprocessing_source_sha256'],
    batch_size=BATCH_SIZE,head_epochs=HEAD_EPOCHS,finetune_epochs=FINETUNE_EPOCHS,patience=PATIENCE,
    head_learning_rate=HEAD_LEARNING_RATE,finetune_learning_rate=FINETUNE_LEARNING_RATE,
    temperature=TEMPERATURE,alpha=ALPHA,threshold=.5,weights='imagenet',dropout=.2,
    supervised_weighting='balanced_train_only',soft_weighting='unweighted',
    loss='(1-alpha)*weighted_binary_CE + alpha*T^2*Bernoulli_KL(teacher||student)',
    augmentation='horizontal_flip_only; paired cached teacher view',
    batch_norm='frozen_non_fused',checkpoint_selection='validation macro_f1; tie positive recall; retain head on cross-stage tie',
    initialization='same ImageNet and classifier initialization for matched control and KD',
    b04_compat_sha256=B04_COMPAT_SHA256,engine_sha256=ENGINE_SHA256,
    tensorflow=tf.__version__,tf_keras=keras.__version__,precision='float32',
    teacher_sources=SOURCES,teacher_cache_sha256={k:file_sha(p/'train_logits.csv') for k,p in CACHE_FOLDERS.items()},
    inbreast_identity_status='filename_patient_ID_documented_separate_audit',
    test_status='B03 previously inspected; exclude test from B05 tuning')
EXPERIMENT_ID=object_sha(PROTOCOL)[:12]
EXPERIMENT_DIR=B05_ROOT/DATASET/EXPERIMENT_ID
EXPERIMENT_DIR.mkdir(parents=True,exist_ok=True)
guarded_json(EXPERIMENT_DIR/'protocol.json',PROTOCOL)
save_json(EXPERIMENT_DIR/'data_provenance.json',DATA_META)
save_json(EXPERIMENT_DIR/'environment.json',environment_record())
AUDIT.to_csv(EXPERIMENT_DIR/'split_audit.csv',index=False)
(EXPERIMENT_DIR/'b04_compat.py').write_text(B04_COMPAT_SOURCE)
(EXPERIMENT_DIR/'b05_hinton_engine.py').write_text(KD_ENGINE_SOURCE)
RUN_PLAN=[(student,'none') for student in STUDENTS_TO_RUN]+[(student,teacher) for teacher in TEACHERS_TO_RUN for student in STUDENTS_TO_RUN]
save_json(EXPERIMENT_DIR/'run_plan.json',dict(runs=RUN_PLAN,teacher_cache_folders={k:str(v) for k,v in CACHE_FOLDERS.items()}))
display(pd.DataFrame(RUN_PLAN,columns=['student','teacher']))
print('Output:',EXPERIMENT_DIR)
print('Run:',len(RUN_PLAN),'| Teachers have not been retrained | Test:',EVALUATE_TEST)

## Cell 7 — Training enam run, masing-masing 30 epoch
Run yang selesai dilewati. Jika terputus, ulangi dari atas dengan konfigurasi sama untuk melanjutkan dari checkpoint epoch terakhir yang tersimpan. Optimizer direset saat berpindah head ke fine-tuning seperti B05 awal.


In [ ]:
RESULTS=[];RUN_FOLDERS=[]
for student,teacher in RUN_PLAN:
    print('TRAIN',DATASET,student,'<-',teacher,'Student seed',TRAINING_SEED,flush=True)
    values=TEACHER_VALUES.get(teacher)
    result,folder=run_experiment(student,teacher,values,FRAMES,IMAGE_DIR,DATA_META,EXPERIMENT_DIR,PROTOCOL,SOURCES)
    if result['epochs_completed']!=HEAD_EPOCHS+FINETUNE_EPOCHS:
        raise RuntimeError('Run belum mencapai 30 epoch')
    RESULTS.append(result);RUN_FOLDERS.append(folder)
    pd.DataFrame(RESULTS).to_csv(EXPERIMENT_DIR/'validation_summary.csv',index=False)
print('Done:',len(RESULTS),'run')

## Cell 8 — Summary of validation and differences compared to the control
A positive delta value indicates that KD performs better on that metric. Do not conclude that KD is always better than a single seed. Rankings apply only within the dataset; labels and cohorts are not identical across datasets.

In [ ]:
summary=pd.DataFrame(RESULTS)
show=['dataset','model','teacher','precision','recall','f1','macro_f1','specificity','accuracy','roc_auc','best_stage','best_epoch','total_train_minutes']
display(summary[show].round(4))
DELTAS=compare_controls(RESULTS,EXPERIMENT_DIR)
display(DELTAS.round(4))
# Verify pairs based on ID, label and initialisation (not CSV order).
for student in STUDENTS_TO_RUN:
    control=EXPERIMENT_DIR/(student+'__from__none')/f'seed_{TRAINING_SEED}'
    initial=read_json(control/'head/initialization.json')
    for teacher in TEACHERS_TO_RUN:
        folder=EXPERIMENT_DIR/(student+'__from__'+teacher)/f'seed_{TRAINING_SEED}'
        if read_json(folder/'head/initialization.json')!=initial:raise RuntimeError('Unmatched Student initialization')
        for p in [control,folder]:
            check_frame(pd.read_csv(p/'validation_predictions.csv',dtype={'sample_id':str,'patient_id':str}),FRAMES['validation'])
cache_summary=pd.DataFrame([dict(teacher=k,**read_json(p/'COMPLETE.json'),cache_folder=str(p)) for k,p in CACHE_FOLDERS.items()])
cache_summary.to_csv(EXPERIMENT_DIR/'teacher_cache_summary.csv',index=False)
display(cache_summary[['teacher','images','views','cache_minutes']].round(3))
print('Metric P/R/F1 is positive; threshold 0.5. Training time excludes the Teacher cache; cache time is listed separately.')

## Cell 9 — Test tetap dinonaktifkan
Ablation ini hanya menggunakan validation untuk pemilihan checkpoint. Tidak mengoptimalkan threshold.


In [ ]:
if EVALUATE_TEST:
    raise RuntimeError('Test tidak boleh dievaluasi dalam ablation ini')
print('Validation-only ablation; tidak ada evaluasi test.')


## Cell 10 — ZIP file of reports for review
The ZIP file contains CSV files of predictions/metrics, the blue confusion matrix, history, Teacher audit, configuration and trainer source code. The model weights and complete logit cache have not been included to keep the ZIP file compact; both remain stored on Drive. The ZIP file can be created once some of the runs have completed.

In [ ]:
REPORT_PATH=EXPERIMENT_DIR/f'B05B_Fixed25_HintonKD_{DATASET}_seed_{TRAINING_SEED}_{EXPERIMENT_ID}_report.zip'
with zipfile.ZipFile(REPORT_PATH,'w',zipfile.ZIP_DEFLATED) as z:
    for p in EXPERIMENT_DIR.rglob('*'):
        if p.is_file() and p.suffix in ['.json','.csv','.png','.py']:
            z.write(p,str(p.relative_to(EXPERIMENT_DIR)))
    for teacher,cache_folder in CACHE_FOLDERS.items():
        for name in ['manifest.json','teacher_audit.json','COMPLETE.json']:
            z.write(cache_folder/name,f'teacher_audit/{teacher}/{name}')
        # B04B validation reference from the verified source, never test.
        source_folder=Path(SOURCES[teacher]['folder'])
        for name in ['validation_metrics.json','config.json','COMPLETE.json']:
            z.write(source_folder/name,f'b04_teacher_reference/{teacher}/{name}')
print('Report:',REPORT_PATH)
print('Run metrics available:',len(list(EXPERIMENT_DIR.glob('*/seed_*/validation_metrics.json'))))
if DOWNLOAD_REPORT:
    from google.colab import files
    files.download(str(REPORT_PATH))

## Melanjutkan eksperimen
- Jalankan dari atas pada Colab GPU dengan konfigurasi sama. Enam Student mulai dari ImageNet saat pertama kali dijalankan; checkpoint B05 awal tidak dilanjutkan.
- Setiap run menyelesaikan 5 epoch head dan 25 epoch fine-tuning. Checkpoint terbaik dapat berasal dari epoch lebih awal; ini tidak berarti training berhenti sebelum waktunya.
- Setelah seluruh run selesai, kirim ZIP B05B dari Cell 10. Hasil B05 awal tetap menjadi pembanding.
- Seed, data, preprocessing, augmentasi, alpha, T, optimizer dan aturan seleksi checkpoint dipertahankan. Perubahan eksperimental adalah penghentian dini yang dinonaktifkan.
- Kegagalan runtime tetap dapat menghentikan proses. Pemeriksaan anggaran epoch mencegah run yang belum lengkap ditandai selesai.
